# 1. Dataset candidates

| Candidate | Source | Samples | Features | License |
|---|---|---|---|---|
| Wine | UCI ML Repository (also in scikit-learn) | 178 | 13 | CC BY 4.0 |
| Breast Cancer Wisconsin (Diagnostic) | UCI ML Repository (also in scikit-learn) | 569 | 30 | CC BY 4.0 |
| Palmer Penguins | Kaggle / allisonhorst/palmerpenguins | 344 | 7 | CC0 |

**Chosen: Wine.** All 13 features are numeric with no missing values, and the 3 classes are well separated. The data is tiny (178 rows), so training is instant and we can run many experiments. No download is needed. Breast Cancer is binary, and Penguins has missing values and mixed types.
Since we're a french team, we chose the Wine dataset because it is related to wine.

# 2. Problem definition

- **Input**: 13 chemical measurements of a wine (alcohol, malic acid, ...)
- **Target**: the grape cultivar (`class_0`, `class_1`, `class_2`)
- **Problem type**: multiclass classification (3 classes)

In [1]:
import pandas as pd
from sklearn.datasets import load_wine

wine = load_wine(as_frame=True)
df = wine.frame
print(df.shape)
df.head()

(178, 14)


,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,target
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0,0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0,0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0,0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0,0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0,0


# 3. Evaluation metric

In [6]:
counts = df["target"].value_counts().sort_index()
pd.DataFrame({
    "class": wine.target_names,
    "samples": counts.values,
    "ratio (%)": (counts / len(df) * 100).round(1).values,
}).set_index("class")

,samples,ratio (%)
class,,
class_0,59,33.1
class_1,71,39.9
class_2,48,27.0


The classes are 59 : 71 : 48 (33% / 40% / 27%): mildly imbalanced, but no class is rare, so **accuracy** a fair metric and easy to read. We also look at the accuracy of each class separately, to check that a model is not good on one class only.

# 4. Common-sense baseline

Always answer with the most frequent class. It gives a reference since a real model is only useful if it clearly beats it.

Always predicting `class_1` gives about 40% accuracy.

In [ ]:
y = df["target"]
majority = y.mode()[0]
pred = pd.Series(majority, index=y.index)
baseline_acc = (pred == y).mean()
print(f"Most frequent class: {wine.target_names[majority]}")
print(f"Baseline accuracy: {baseline_acc:.3f}")

Most frequent class: class_1
Baseline accuracy: 0.399


**Accuracy per class**: for each type of wine, how many did the baseline get right? It only knows `class_1`, so it is 100% right on `class_1` and 0% right on the others.

In [4]:
per_class = pd.DataFrame({
    "class": wine.target_names,
    "wines": y.value_counts().sort_index().values,
    "correct": (pred == y).groupby(y).sum().values,
})
per_class["accuracy (%)"] = (per_class["correct"] / per_class["wines"] * 100).round(1)
per_class.set_index("class")

,wines,correct,accuracy (%)
class,,,
class_0,59,0,0.0
class_1,71,71,100.0
class_2,48,0,0.0
